### 00. import dependencies

In [1]:
import os
import os.path
import warnings
from dotenv import load_dotenv
from typing import TypedDict, Annotated



# for routing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# langchain
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AnyMessage, SystemMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

# langgraph
from langgraph.graph import StateGraph, START, END,add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

# DB + ORM
from sqlalchemy import create_engine,Column,Integer,String
from sqlalchemy.orm import sessionmaker,declarative_base

# vector DB
import chromadb

# GMAIL API
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from email.message import EmailMessage
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
import base64

warnings.filterwarnings("ignore")


/home/breezy/projects for int/agentic_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/breezy/projects for int/agentic_RAG/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### 01. Creating the Environment variables

In [2]:
#### Load the api_key

load_dotenv(dotenv_path='../.env')

API_KEY = os.getenv("GOOGLE_API_KEY")
DB_URL = os.getenv("DB_URI")
DEFAULT_GEMINI_MODEL = "gemini-2.5-flash"

if not API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

if not DB_URL:
    raise ValueError("DB_URL not found in environment variables")

In [3]:
#### initialize the models

llm = ChatGoogleGenerativeAI(model=DEFAULT_GEMINI_MODEL, temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="model/embedding-001")

### 02. RAG pipline

In [4]:
### initialize ChromaDB

chroma_client = chromadb.Client()
collection_name = "enterprise_hr_collection"

try:
    chroma_client.delete_collection(collection_name)
except Exception as e:
    print(f"There no collection to delete : {e}")
collection = chroma_client.create_collection(collection_name)

In [5]:
### Load the documents

file_paths = [
    "../data/raw/hr_policy.pdf",
    "../data/raw/complaints_policy.pdf",
    "../data/raw/leave_policy.pdf"
]

documents = []
for path in file_paths:
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        documents.extend(loader.load())
    else:
        print(f"Warning: {path} not found. Skipping.")

In [6]:
### Chunking and embedding

if documents:
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ".", " "],
        chunk_size=400,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False,
    )
    chunks =text_splitter.split_documents(documents)
    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    ids = [f"doc_{i}" for i in range(len(chunks))]

    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    print(f"Added {len(texts)} chunks into chromaDB.")
else:
    print("No documents to add.")

Added 17 chunks into chromaDB.


### 03. database setup (sample)

In [7]:
Base =  declarative_base()

class EmployeeModel(Base):
    __tablename__ = "employees"
    employee_id = Column(String, primary_key=True)
    name = Column(String)
    pto_days = Column(Integer)
    sick_days = Column(Integer)
    email = Column(String)

engine = create_engine(DB_URL)
SessionLocal= sessionmaker(bind=engine)

Base.metadata.create_all(bind=engine)

### 04. Creating tools

In [ ]:
### connect to the gmail api with credentials

SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def get_gmail_service():
    """Gets valid user credentials from storage or prompts the user."""
    creds = None
    ### check the token is available in local machine or not
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    ### if creds is not have or not valid then create a token and save to token.json
    if not creds or not creds.valid:

        if creds and creds.expired and creds.refresh_token:
            ## Try refreshing login
            creds.refresh(Request())
        else:
            ## if refresh not possible, ask user again to log in
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    return build('gmail', 'v1', credentials=creds)



In [8]:
@tool
def check_sql_leave_balance(employee_id:str)-> str:
    """Check PTO and sick leave balance, name, and email for a given employee ID."""
    db = SessionLocal()
    try:
        emp = db.query(EmployeeModel).filter_by(employee_id = employee_id).first()
        if not emp:
            return f"Employee {employee_id} not found."

        # UPDATE THIS LINE to include email and ID
        return f"Name: {emp.name}, Email: {emp.email}, ID: {employee_id}. Balance: {emp.pto_days} PTO, {emp.sick_days} sick leaves."
    finally:
        db.close()


@tool
def hr_vector_search(query:str)->str:
    """Search HR policy documents using semantic vector search."""
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    if results['documents'] and results['documents'][0]:
        docs = results["documents"][0]
        metas = results["metadatas"][0]

        formatted_context = ""

        for i in range(len(docs)):
            source_path = metas[i].get('source', 'Unknown Document')
            filename = os.path.basename(source_path)
            page_num = metas[i].get('page', -1) + 1

            formatted_context += f"--- SOURCE: {filename} (Page {page_num}) ---\n{docs[i]}\n\n"
        return formatted_context

    return "No relevant policy found."

@tool
def draft_and_send_email(recipient: str, subject: str, body: str, is_confirmed: bool = False) -> str:
    """Draft an email and send it only after user confirmation."""

    # 1. The Draft Phase
    if not is_confirmed:
        return f"""
EMAIL_DRAFT:
To: {recipient}
Subject: {subject}

{body}

Reply YES to confirm sending.
"""

    # 2. The Sending Phase (executes only when is_confirmed=True)
    try:
        # Get the authenticated Gmail service
        service = get_gmail_service()

        # Construct the MIME email
        message = EmailMessage()
        message.set_content(body)
        message['To'] = recipient
        message['Subject'] = subject

        # Encode the message securely
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
        create_message = {'raw': encoded_message}

        # Send it!
        service.users().messages().send(userId="me", body=create_message).execute()

        print(f"\n[ACTUAL EMAIL SENT] → Successfully sent to {recipient}")
        return "SUCCESS: Email sent via your personal Gmail account."

    except Exception as e:
        print(f"\n[EMAIL FAILED] → {e}")
        return f"FAILED to send email: {str(e)}"

### 05. Create router layer 1

In [9]:
### making state
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    user_context: dict

In [10]:
intent_profiles = {
    "leave_agent" : ["leave","vacation","pto","holiday"],
    "complaint_agent" : ["complaint","issue","harassment"],
    "hr_agent":["policy","salary","benefits","dress code"]
}

def semantic_router(state: State):
    messages = state.get("messages", [])

    # Helper function to safely extract text from LangChain message content
    def safe_extract_text(content):
        if isinstance(content, list):
            return " ".join([item.get("text", "") for item in content if isinstance(item, dict)]).lower()
        return str(content).lower()

    # 1. Grab the current user text safely
    user_text = ""
    for msg in reversed(messages):
        if getattr(msg, 'type', '') == 'human':
            user_text = safe_extract_text(msg.content).strip()
            break

    if not user_text:
        return Command(goto="hr_agent")

    # 2. Check for active conversation flow (Look at the LAST AI message)
    history_to_check = messages[:-1] if len(messages) > 1 else []
    last_ai_msg = ""
    for msg in reversed(history_to_check):
        if getattr(msg, 'type', '') == 'ai':
            last_ai_msg = safe_extract_text(msg.content)
            break # Found the most recent AI response

    # If the AI was just talking about drafting an email, stay with the COMM_AGENT
    if "reason for" in last_ai_msg or "draft" in last_ai_msg or "reply yes" in last_ai_msg:
        print("[Router] Active email drafting flow detected! Routing directly to -> COMM_AGENT")
        return Command(goto="comm_agent")

    # 3. Robust Keyword Routing
    print(f"[Router] Evaluating user input: '{user_text}'")
    for agent, keywords in intent_profiles.items():
        if any(keyword in user_text for keyword in keywords):
            print(f"[Router] Intent matched! Routing to -> {agent.upper()}")
            return Command(goto=agent)

    # return to the HR_Agent
    print("[Router] No keywords matched. Defaulting to -> HR_AGENT")
    return Command(goto="hr_agent")

In [11]:
### 06. making Agent (Native LangGraph)
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

LEAVE_PROMPT = "You are the Enterprise Leave Agent. Answer leave queries using your tools."

COMM_PROMPT = """
You are the Enterprise Communication Agent.
Your job:
1. Understand user's request
2. Draft a professional email
3. ALWAYS call the tool draft_and_send_email
Rules:
- Extract recipient, subject, body clearly
- First call: is_confirmed=False
- If the user says YES to confirm sending -> call again with is_confirmed=True
"""


# Custom factory to build an agent without deprecated functions
def build_native_agent(llm, tools, system_prompt=None):
    # 1. Bind the tools directly to the LLM
    llm_with_tools = llm.bind_tools(tools)

    # 2. Define the node that calls the model
    def call_model(state: State):
        messages = state.get("messages", [])

        # Inject system prompt if it exists
        if system_prompt:
            messages = [SystemMessage(content=system_prompt)] + messages

        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    # 3. Build the agent's internal workflow
    workflow = StateGraph(State)
    workflow.add_node("agent", call_model)
    workflow.add_node("tools", ToolNode(tools))  # ToolNode handles tool execution safely

    # 4. Define the routing: Loop tools if needed, otherwise finish
    workflow.add_edge(START, "agent")
    workflow.add_conditional_edges("agent", tools_condition)
    workflow.add_edge("tools", "agent")

    return workflow.compile()


# Instantiate our sub-graphs
leave_agent = build_native_agent(llm, [check_sql_leave_balance, hr_vector_search], LEAVE_PROMPT)
complaint_agent = build_native_agent(llm, [hr_vector_search])
hr_agent = build_native_agent(llm, [hr_vector_search])
comm_agent = build_native_agent(llm, [draft_and_send_email], COMM_PROMPT)

### 06. making Agent

In [12]:
LEAVE_PROMPT = """
You are the Enterprise Leave Agent.
Answer leave queries using your tools.
If the user asks you to email, notify, or message someone → DO NOT draft it yourself.
Instead, output exactly this phrase: HANDOFF_TO_COMMUNICATION
"""
COMM_PROMPT = """
You are the Enterprise Communication Agent.
Draft the email or notification.
Always use your tools, but remember the email tool requires confirmation first.
"""

leave_agent = create_agent(
    model=llm,
    tools=[check_sql_leave_balance,hr_vector_search],
    system_prompt=LEAVE_PROMPT)

complaint_agent = create_agent(
    model=llm,
    tools=[hr_vector_search]
)
hr_agent = create_agent(
    model=llm,
    tools=[hr_vector_search]
)
comm_agent = create_agent(
    model=llm,
    tools=[draft_and_send_email],
    system_prompt=COMM_PROMPT
)

### 07. making node logics

In [13]:
def run_leave_agent(state:State):
    result = leave_agent.invoke(state)
    user_text = state["messages"][-1].content.lower()

    if any(word in user_text for word in ["email", "send", "notify", "message"]):
        print("[Rule Override] Forcing handoff to Communication Agent...")
        return Command(update={"messages": result["messages"]}, goto="comm_agent")

    return Command(update= {"messages": result["messages"]},goto=END)

def run_hr_agent(state:State):
    result = hr_agent.invoke(state)
    return Command(update= {"messages": result["messages"]},goto=END)

def run_complaint_agent(state:State):
    result = complaint_agent.invoke(state)
    return Command(update= {"messages": result["messages"]},goto=END)

def run_comm_agent(state:State):
    result = comm_agent.invoke(state)
    return Command(update= {"messages": result["messages"]},goto=END)

In [14]:
### 07. making node logics
def run_leave_agent(state: State):
    result = leave_agent.invoke(state)
    return Command(update={"messages": result["messages"]}, goto=END)


def run_hr_agent(state: State):
    result = hr_agent.invoke(state)
    return Command(update={"messages": result["messages"]}, goto=END)


def run_complaint_agent(state: State):
    result = complaint_agent.invoke(state)
    return Command(update={"messages": result["messages"]}, goto=END)


def run_comm_agent(state: State):
    result = comm_agent.invoke(state)
    return Command(update={"messages": result["messages"]}, goto=END)

### 08. context injection (add the employee id )

In [15]:
# def inject_context(state:State):
#     state["user_context"] = {"employee_id": "EMP001"}
#     return state
def inject_context(state: State):
    messages = state.get("messages", [])

    if len(messages) == 1:
        system_note = SystemMessage(content="System Note: The current user's Employee ID is EMP001. Do not ask for their ID, just use this one.")

        return {
            "user_context": {"employee_id": "EMP001"},
            "messages": [system_note]
        }

    return {"user_context": {"employee_id": "EMP001"}}

In [16]:
builder = StateGraph(State)

builder.add_node("context", inject_context)
builder.add_node("router", semantic_router)
builder.add_node("leave_agent", run_leave_agent)
builder.add_node("complaint_agent", run_complaint_agent)
builder.add_node("hr_agent", run_hr_agent)
builder.add_node("comm_agent", run_comm_agent)

builder.add_edge(START,"context")
builder.add_edge("context","router")

app = builder.compile(checkpointer=MemorySaver())

In [17]:
config = {"configurable": {"thread_id": "user_1"}}


def chat(user_input):
    events = app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="values"
    )

    final = None
    for e in events:
        final = e["messages"][-1]

    output = final.content


    if isinstance(output, list):
        clean_text = " ".join([item["text"] for item in output if item.get("type") == "text"])
    else:
        clean_text = output

    print(f"\n🤖: {clean_text}")


chat("How many leave days do I have?")

[Router] Low confidence. Routing to -> HR_AGENT

🤖: I cannot tell you how many leave days you have using this tool. I can only search HR policy documents. You would need to check with your HR department or a dedicated HR system for that information.


In [ ]:
chat("Send an email requesting leave tomorrow")